# SSD VOC Detection
In this notebook, we are going to train SSD for later epochs.

In [1]:
!pip install git+https://github.com/obsessor-ak1/torch-ignite.git@obj_detection_bugfix

  Cloning https://github.com/obsessor-ak1/torch-ignite.git (to revision obj_detection_bugfix) to /tmp/pip-req-build-v78mlhlm
  Running command git clone --filter=blob:none --quiet https://github.com/obsessor-ak1/torch-ignite.git /tmp/pip-req-build-v78mlhlm
  Running command git checkout -b obj_detection_bugfix --track origin/obj_detection_bugfix
  Switched to a new branch 'obj_detection_bugfix'
  Branch 'obj_detection_bugfix' set up to track remote branch 'obj_detection_bugfix' from 'origin'.
  Resolved https://github.com/obsessor-ak1/torch-ignite.git to commit 9fa3c53cd84f51ffc88a6f116bef7cb35f580dca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pytorch-ignite: filename=pytorch_ignite-0.6.0-py3-none-any.whl size=344794 sha256=e98d3aba00030952e0801aa15a43b4ebeb5f1a36f379ccdc0c31965c839dea79
  Stored in directory: /tmp/pip-ephem-wheel-cache-fbhfetnw/wheels/22/78/11/9480d8f07c8428

In [2]:
!pip install git+https://github.com/obsessor-ak1/Object_Detection.git --no-deps

  Cloning https://github.com/obsessor-ak1/Object_Detection.git to /tmp/pip-req-build-ge66yv_l
  Running command git clone --filter=blob:none --quiet https://github.com/obsessor-ak1/Object_Detection.git /tmp/pip-req-build-ge66yv_l
  Resolved https://github.com/obsessor-ak1/Object_Detection.git to commit e52460007a0047f440072b842df3775660b4bd86
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for object_detection: filename=object_detection-0.0.1-py3-none-any.whl size=9383 sha256=07f424c6eadfc18d1c15874f0e16601b77fa9ee4dc6fa1e570f9b9b8a9d058ab
  Stored in directory: /tmp/pip-ephem-wheel-cache-aha_oelg/wheels/db/1c/1b/130528d9f2d8cd8c38fbe5fd63c3bf2014476a86eaef0eac35
Successfully built object_detection


In [3]:
from typing import Optional
import os

import torch
from torch import nn
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision import tv_tensors

from ignite.engine import Engine, Events
from ignite.handlers import global_step_from_engine
from ignite.handlers.checkpoint import Checkpoint, DiskSaver
from ignite.handlers.tqdm_logger import ProgressBar
from ignite.handlers.wandb_logger import WandBLogger, OutputHandler
from ignite.metrics import Average, ObjectDetectionAvgPrecisionRecall, RunningAverage
import kagglehub

from detection_tools.data.pascal_voc import SSDVOCDataset
from detection_tools.ssd.utils import AnchorGenerator, Matcher, OffsetHandler, SSDPredictor
from detection_tools.ssd.modules import SSDLoss, SSDHead, VGG16FeatureExtractor

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ak_chp1 (ak_chp1-panjab-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
class SSDDetector(nn.Module):
    def __init__(self, num_classes_actual: int, device: torch.device = torch.device("cpu")):
        super().__init__()
        self.feature_extractor = VGG16FeatureExtractor()
        self.device = device
        self.anchor_generator = AnchorGenerator(
            aspect_ratios=[
                [1.0, 2.0],
                [1.0, 2.0, 3.0],
                [1.0, 2.0, 3.0],
                [1.0, 2.0, 3.0],
                [1.0, 2.0],
                [1.0, 2.0]
            ],
            device=device
        )
        self.matcher = Matcher(iou_threshold=0.5)
        self.offset_handler = OffsetHandler()
        self.head = SSDHead(
            num_classes=num_classes_actual+1, # +1 for background class
            num_anchors=self.anchor_generator.num_anchors,
            channels=self.feature_extractor.MAP_CHANNELS
        )
        self.loss = SSDLoss(o_handler=self.offset_handler)
        self.predictor = SSDPredictor(
            score_thresh=0.01,
            num_top_k=200,
            nms_thresh=0.45,
            max_detections=100,
            device=device
        )
        self.to(device)
    
    def forward(self, images: torch.Tensor, targets: Optional[list[dict[str, torch.Tensor]]] = None) -> dict[str, torch.Tensor]:
        """If training targets are required and the model returns the loss, else returns the predictions."""
        img_h, img_w = images.shape[-2:]

        anchors = self.anchor_generator.generate_anchors(
            image_size=(img_h, img_w),
            feature_map_sizes=self.feature_extractor.MAP_SHAPES_300
        )
        anchors = [anchors.clone() for _ in range(images.shape[0])]
        features = self.feature_extractor(images)
        head_outputs = self.head(features)
        if targets is not None:
            matches = [
                self.matcher(target["bbox"], anchor_set)
                for target, anchor_set in zip(targets, anchors)
            ]
        
        if self.training:
            loss = self.loss(targets, head_outputs, anchors, matches)
            return loss
        else:
            predictions = self.predictor(head_outputs, anchors)
            if targets is not None:
                val_loss = self.loss(targets, head_outputs, anchors, matches, raw=True)
                return predictions, targets, val_loss
            return predictions
        

In [7]:
model = SSDDetector(num_classes_actual=20, device=device)
model

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 228MB/s]


SSDDetector(
  (feature_extractor): VGG16FeatureExtractor(
    (features_conv4_3): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   

In [8]:
train_transforms = v2.Compose([
    v2.ToImage(),
    v2.RandomPhotometricDistort(p=0.5),
    v2.RandomZoomOut(fill={tv_tensors.Image: 127}),
    v2.RandomIoUCrop(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.Resize((300, 300), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(size=(300, 300), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [9]:
!ls /kaggle/input/notebooks/ak11chp/ssd-pascal-voc-detection/checkpoints

'ssd_detector_voc_checkpoint_67_mAP@50=0.0916.pt'
'ssd_detector_voc_checkpoint_75_mAP@50=0.0930.pt'


In [10]:
full_data_path = "/kaggle/input/datasets/ak11chp/pascal-voc-dataset/Pascal_VOC"
full_checkpoints_path = "/kaggle/input/notebooks/ak11chp/ssd-pascal-voc-detection/checkpoints/ssd_detector_voc_checkpoint_75_mAP@50=0.0930.pt"

In [11]:
train_set = SSDVOCDataset(
    root=full_data_path,
    image_set="train",
    transforms=train_transforms
)
val_set = SSDVOCDataset(
    root=full_data_path,
    image_set="val",
    transforms=val_transform
)
print(f"Train set size: {len(train_set)}, Validation set size: {len(val_set)}")

Train set size: 5717, Validation set size: 5823


In [12]:
def collate_fn(batch):
    images = [sample[0] for sample in batch]
    targets = [sample[1] for sample in batch]
    images = torch.stack(images)
    return images, targets

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [13]:
optimizer = torch.optim.SGD([
    {"params": model.feature_extractor.parameters(), "lr": 1e-5},
    {"params": model.head.parameters(), "lr": 1e-3}],
    momentum=0.9,
    weight_decay=5e-4
)

In [14]:
def train_batch(engine, batch):
    model.train()
    optimizer.zero_grad()
    images, targets = batch
    images = images.to(device)
    targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
    loss = model(images, targets)
    loss.backward()
    optimizer.step()
    return loss.item()

def validate_batch(engine, batch):
    model.eval()
    images, targets = batch
    images = images.to(device)
    targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
    with torch.no_grad():
        output = model(images, targets)
    return output

In [15]:
trainer = Engine(train_batch)
validator = Engine(validate_batch)

In [16]:
train_metrics = {
    "loss": RunningAverage(output_transform=lambda x: x),
}
def output_transform_mAP(output):
    preds = output[0]
    targets = output[1]
    # preds = [{k: v.cpu() for k, v in pred.items()} for pred in preds]
    # targets = [{k: v.cpu() for k, v in target.items()} for target in targets]
    return preds, targets
    
mAP_metric = ObjectDetectionAvgPrecisionRecall(
    num_classes=21,
    output_transform=output_transform_mAP,
    iou_thresholds=[0.5],
    device=device
)
val_metrics = {
    "reg_loss": Average(output_transform=lambda output: output[2]["reg_loss"]),
    "cls_loss": Average(output_transform=lambda output: output[2]["cls_loss"]),
    "full_loss": Average(output_transform=lambda output: output[2]["reg_loss"] + output[2]["cls_loss"]),
    "mAP": mAP_metric
}

In [17]:
for key, metric in train_metrics.items():
    metric.attach(trainer, name=key)
for key, metric in val_metrics.items():
    metric.attach(validator, name=key)

In [18]:
scheduler = MultiStepLR(optimizer, milestones=[40, 80], gamma=0.1)
@trainer.on(Events.EPOCH_COMPLETED)
def step_scheduler(engine):
    scheduler.step()
    # Optional: print the current LR to verify it dropped!
    backbone_lr = optimizer.param_groups[0]["lr"]
    current_lr = optimizer.param_groups[1]['lr']
    print(f"Current Head LR: {current_lr}")
    print(f"Current backbone LR: {backbone_lr}")

In [19]:
pg_bar1 = ProgressBar(persist=True, desc="Training")
pg_bar1.attach(trainer, metric_names="all")

@trainer.on(Events.EPOCH_COMPLETED)
def log_final_loss(engine):
    current_epoch = engine.state.epoch
    max_epochs = engine.state.max_epochs
    loss = engine.state.metrics["loss"]
    print(f"Epoch: {current_epoch}/{max_epochs}, Final Loss: {loss:.4f}")
    validator.run(val_loader)

In [20]:
pg_bar2 = ProgressBar(persist=True, desc="Validating")
pg_bar2.attach(validator, output_transform=lambda output: output[2])

@validator.on(Events.EPOCH_COMPLETED)
def log_validation_results(engine):
    current_epoch = trainer.state.epoch
    max_epochs = trainer.state.max_epochs
    reg_loss = engine.state.metrics["reg_loss"]
    cls_loss = engine.state.metrics["cls_loss"]
    full_loss = engine.state.metrics["full_loss"]
    mAP = engine.state.metrics["mAP"]
    mAP50 = mAP[0]
    print(f"Validation Epoch: {current_epoch}/{max_epochs}: Reg Loss: {reg_loss:.4f}, Cls Loss: {cls_loss:.4f}")
    print(f"Validation Full Loss: {full_loss:.4f} mAP@50: {mAP50:.4f}")

In [21]:
# Loding checkpoints
to_load = {
    "model": model,
    "optimizer": optimizer
}
Checkpoint.load_objects(to_load=to_load, checkpoint=full_checkpoints_path)

In [22]:
# Checkpoints and Loggers
to_save = {
    "model": model,
    "optimizer": optimizer
}
file_name_prefix = "ssd_detector_voc"
def score_function(engine):
    return engine.state.metrics["mAP"][0]
score_name = "mAP@50"
n_saved = 2
step_source = global_step_from_engine(trainer, Events.EPOCH_COMPLETED)
ckp_handler = Checkpoint(
    to_save=to_save,
    save_handler=DiskSaver("./checkpoints", create_dir=True),
    filename_prefix=file_name_prefix,
    score_function=score_function,
    score_name=score_name,
    n_saved=n_saved,
    global_step_transform=step_source
)
validator.add_event_handler(Events.EPOCH_COMPLETED, ckp_handler)

In [23]:
logger = WandBLogger(
    project="SSD_Object_Detection_VOC",
    config={
        "backbone_lr": 1e-5,
        "head_lr": 1e-3,
        "max_epochs": 10
    }
)

logger.attach(
    trainer,
    log_handler=OutputHandler(
        tag="training",
        metric_names="all",
        output_transform=lambda _: None,
        global_step_transform=lambda engine, _: engine.state.epoch
    ),
    event_name=Events.EPOCH_COMPLETED
)


logger.attach(
    validator,
    log_handler=OutputHandler(
        tag="validation",
        metric_names="all",
        output_transform=lambda _: None,
        global_step_transform=step_source
    ),
    event_name=Events.EPOCH_COMPLETED
)

wandb: setting up run 4k66mqrv
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260222_104958-4k66mqrv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vivid-leaf-20
wandb: ⭐️ View project at https://wandb.ai/ak_chp1-panjab-university/SSD_Object_Detection_VOC
wandb: 🚀 View run at https://wandb.ai/ak_chp1-panjab-university/SSD_Object_Detection_VOC/runs/4k66mqrv


In [24]:
trainer.run(train_loader, max_epochs=80)

Training[1/179]   1%|           [00:00<?]

Current Head LR: 0.001
Current backbone LR: 1e-05
Epoch: 1/80, Final Loss: 4.0861


Validating[1/182]   1%|           [00:00<?]

/usr/local/lib/python3.12/dist-packages/ignite/distributed/utils.py:396: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  all_padded_tensors[


Validation Epoch: 1/80: Reg Loss: 0.1000, Cls Loss: 4.0505
Validation Full Loss: 4.1506 mAP@50: 0.0859


...

Training[1/179]   1%|           [00:00<?]

Current Head LR: 1e-05
Current backbone LR: 1.0000000000000002e-07
Epoch: 80/80, Final Loss: 3.7589


Validating[1/182]   1%|           [00:00<?]

Validation Epoch: 80/80: Reg Loss: 0.0859, Cls Loss: 3.7513
Validation Full Loss: 3.8372 mAP@50: 0.1311


State:
	iteration: 14320
	epoch: 80
	epoch_length: 179
	max_epochs: 80
	max_iters: <class 'NoneType'>
	output: 3.479680061340332
	batch: <class 'tuple'>
	metrics: <class 'dict'>
	dataloader: <class 'torch.utils.data.dataloader.DataLoader'>
	seed: <class 'NoneType'>
	times: <class 'dict'>